In [1]:
proximidade <- function(qt, dt, janela = 3) {        # qt: tokens da consulta; dt: tokens do documento
  s <- 0                                             # o escore comeca em zero
  for (i in seq_along(qt)) for (j in seq_along(qt))  # todos os pares de posicoes da consulta...
    if (i < j && qt[i] != qt[j]) {                   # ...com i antes de j e termos diferentes
      pos_i <- which(dt == qt[i])                    # onde o termo i aparece no documento
      pos_j <- which(dt == qt[j])                    # onde o termo j aparece
      if (length(pos_i) && length(pos_j)) {          # os dois aparecem?
        d <- outer(pos_j, pos_i, "-")                # todas as distancias (j menos i)
        d <- d[d > 0 & d <= janela]                  # so j DEPOIS de i, e dentro da janela
        if (length(d)) s <- s + 1 / min(d)           # soma 1 / a menor distancia
      }
    }
  s                                                  # devolve o escore
}
reordenar <- function(rk, k, sc) {               # rk: ranking do estagio 1; sc: escores do estagio 2
  k <- min(k, length(rk))                        # protecao: k nao passa do numero de documentos
  top <- names(rk)[1:k]                          # a cabeca: os k primeiros
  nova <- top[order(-sc[top], seq_along(top))]   # pelo escore, decrescente; empate: ordem do estagio 1
  c(nova, names(rk)[-(1:k)])                     # cabeca reordenada + cauda intacta (NOMES)
}
proximidade(c("a", "b"), c("a", "x", "b"))                         # teste: "b" 2 depois de "a" -> 0.5
reordenar(c(a = 3, b = 2, c = 1), 2, c(a = 0.1, b = 0.9, c = 5))   # teste: tem que dar "b" "a" "c"


[1] 0.5

[1] "b" "a" "c"

In [2]:
estagio1 <- function(texto, ix, cfg, m) {                  # o ranking que o motor do grupo da hoje
  rb <- ranking_bm25(texto, ix, cfg)                       # o BM25, com k1 e b do grupo
  if (cfg$modelo == "bm25") return(rb)                     # modelo "bm25": pronto
  rl <- ranking_lsa(texto, ix, cfg, m)                     # o LSA da Aula 07 (m = lsa(ix, k))
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)   # "lsa", ou o hibrido pela RRF
}                                                          # fim da funcao
m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)   # o LSA so e calculado se for usado

rerank_prox <- function(rk, qt, tokens, k, janela) {    # rk: estagio 1; qt: tokens da consulta
  if (k == 0) return(names(rk))                         # k = 0: nao reordena
  top <- head(names(rk), k)                             # os candidatos (head corta se k > N)
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))   # so eles sao lidos
  reordenar(rk, k, sc)                                  # cabeca reordenada + cauda
}                                                       # fim da funcao

ERROR: Error: object 'cfg' not found


In [3]:
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")
cfg <- tryCatch({
  download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R", quiet = TRUE)
  source("config.R"); cfg
}, error = function(e) {
  cfg_aula()
})
cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"), stringsAsFactors = FALSE, fileEncoding = "UTF-8")
docs   <- setNames(cp$texto, cp$id)
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))
ix     <- montar(docs, cfg)

In [4]:
arqs <- c("qrels_Andre_2026-10-25.csv")
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)
qrels <- ler_qrels(arqs, juiz = cfg$juizes)
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"), stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
consultas <- setNames(nec$texto_consulta, nec$consulta)
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]

In [5]:
q  <- usar[1]
rk <- estagio1(consultas[[q]], ix, cfg, m)
qt <- preparar(consultas[[q]], cfg)
rn <- rerank_prox(rk, qt, ix$tokens, 10, 3)
rbind(estagio1 = names(rk)[1:5], rerank = rn[1:5])

ERROR: Error in rerank_prox(rk, qt, ix$tokens, 10, 3): could not find function "rerank_prox"


In [6]:
proximidade <- function(qt

ERROR: Error in parse(text = input): <text>:2:0: unexpected end of input
1: proximidade <- function(qt
   ^


In [7]:
proximidade <- function(qt, dt, janela = 3) {
  s <- 0
  for (i in seq_along(qt)) for (j in seq_along(qt))
    if (i < j && qt[i] != qt[j]) {
      pos_i <- which(dt == qt[i])
      pos_j <- which(dt == qt[j])
      if (length(pos_i) && length(pos_j)) {
        d <- outer(pos_j, pos_i, "-")
        d <- d[d > 0 & d <= janela]
        if (length(d)) s <- s + 1 / min(d)
      }
    }
  s
}

reordenar <- function(rk, k, sc) {
  k <- min(k, length(rk))
  top <- names(rk)[1:k]
  nova <- top[order(-sc[top], seq_along(top))]
  c(nova, names(rk)[-(1:k)])
}

estagio1 <- function(texto, ix, cfg, m) {
  rb <- ranking_bm25(texto, ix, cfg)
  if (cfg$modelo == "bm25") return(rb)
  rl <- ranking_lsa(texto, ix, cfg, m)
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)
}
m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)

rerank_prox <- function(rk, qt, tokens, k, janela) {
  if (k == 0) return(names(rk))
  top <- head(names(rk), k)
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))
  reordenar(rk, k, sc)
}

In [8]:
q  <- usar[1]
rk <- estagio1(consultas[[q]], ix, cfg, m)
qt <- preparar(consultas[[q]], cfg)
rn <- rerank_prox(rk, qt, ix$tokens, 10, 3)
rbind(estagio1 = names(rk)[1:5], rerank = rn[1:5])

estagio1,d6,d2,d4,d1,d3
rerank,d6,d2,d4,d1,d3


In [9]:
ap_rerank <- function(rk, qt, grau, tokens, k, janela, limiar)               # AP de UMA consulta
  metricas(rerank_prox(rk, qt, tokens, k, janela), grau, limiar)[["AP"]]     # o AP da ordem final

dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])                # consultas de desenvolvimento
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])              # consultas de teste

rks <- lapply(consultas[usar], function(texto) estagio1(texto, ix, cfg, m))  # estagio 1 de cada consulta
qts <- lapply(consultas[usar], function(texto) preparar(texto, cfg))         # tokens limpos de cada consulta

# MAP do estagio 1 isolado (k = 0):
map0 <- mean(sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 0, 3, cfg$limiar)))
round(map0, 3)

# Grade de combinacoes no dev:
grade <- sapply(c(2, 3, 5), function(w)
  sapply(c(3, 5, 10), function(k)
    mean(sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, k, w, cfg$limiar)))))
rownames(grade) <- c("k3", "k5", "k10")
colnames(grade) <- c("janela2", "janela3", "janela5")
round(grade, 3)

[1] 0.066

,janela2,janela3,janela5
k3,0.066,0.066,0.066
k5,0.066,0.066,0.066
k10,0.066,0.066,0.066


In [10]:
ap0 <- sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 0, 3, cfg$limiar))   # estagio 1
ap1 <- sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 5, 3, cfg$limiar))   # rerank k=5, janela 3
round(rbind(estagio1 = ap0, rerank = ap1), 3)
dev[ap1 < ap0]                                                    # onde perdeu

,q01,q02,q03
estagio1,0.023,0.097,0.079
rerank,0.023,0.097,0.079


character(0)

In [11]:
k_esc <- 0; j_esc <- 3
tab_teste <- sapply(teste, function(q)
  sapply(c(estagio1 = 0, rerank = k_esc), function(k)
    ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, k, j_esc, cfg$limiar)))
round(tab_teste, 3); round(rowMeans(tab_teste), 3)

,q04,q05
estagio1,0.112,0.071
rerank,0.112,0.071


estagio1   rerank 
   0.092    0.092

In [12]:
list.files()


[1] "00_FICHA_PROJETO.md"          "aula08_parteD_consolidado.md"
[3] "config.R"                     "qrels_Andre_2026-10-25.csv"  
[5] "sample_data"

In [13]:
anexar_estado("00_FICHA_PROJETO.md")
anexar_estado("aula08_parteD_consolidado.md")

Estado do R anexado a 00_FICHA_PROJETO.md (65 linhas de objetos).

Estado do R anexado a aula08_parteD_consolidado.md (65 linhas de objetos).

